# Chargement du dataset

In [2]:
import pandas as pd
import os

# 1. Définition du chemin relatif depuis le dossier notebooks/
# Remplace les '...' par la fin exacte du nom de ton fichier si besoin
file_name = "top_1000_most_swapped_books.csv" 
raw_data_path = os.path.join("..", "data", "raw", file_name)

# 2. Chargement du dataset
try:
    df_books = pd.read_csv(raw_data_path)
    print("✅ Dataset chargé avec succès !")
    print(f"Dimensions du dataset : {df_books.shape[0]} lignes et {df_books.shape[1]} colonnes.\n")
except FileNotFoundError:
    print(f"❌ Erreur : Le fichier n'a pas été trouvé au chemin : {raw_data_path}")
    print("Vérifie bien l'extension et l'orthographe exacte du nom du fichier.")

# 3. Premier aperçu des données
df_books.info()

✅ Dataset chargé avec succès !
Dimensions du dataset : 990 lignes et 18 colonnes.

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 990 entries, 0 to 989
Data columns (total 18 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   id                    990 non-null    int64  
 1   title                 990 non-null    object 
 2   author                990 non-null    object 
 3   genre                 990 non-null    object 
 4   language              990 non-null    object 
 5   publicationYear       990 non-null    int64  
 6   publisher             990 non-null    object 
 7   description           990 non-null    object 
 8   pageCount             990 non-null    int64  
 9   tags                  990 non-null    object 
 10  rating_average        990 non-null    float64
 11  most_popular_country  990 non-null    object 
 12  bestseller_status     990 non-null    bool   
 13  awards                384 non-null    obje

In [7]:
df_books.head()

,id,title,author,genre,language,publicationYear,publisher,description,pageCount,tags,rating_average,most_popular_country,bestseller_status,awards,age_category,adapted_to_movie,movie_release_year,isbn
0,1,Harry Potter and the Sorcerer's Stone,J.K. Rowling,Fantasy,English,1997,Bloomsbury,A young wizard discovers his magical heritage ...,309,"magic,school,adventure",4.89,UK,True,"Smarties Prize,British Book Award",Children,True,2001.0,978-0747532743
1,2,To Kill a Mockingbird,Harper Lee,Southern Gothic,English,1960,J.B. Lippincott & Co.,A lawyer in the Depression-era South defends a...,281,"classic,law,racism,history",4.85,USA,True,Pulitzer Prize,Adult,True,1962.0,978-0061120084
2,3,1984,George Orwell,Dystopian,English,1949,Secker & Warburg,A dystopian social science fiction novel and c...,328,"politics,scifi,totalitarianism",4.80,UK,True,Prometheus Hall of Fame,Adult,True,1984.0,978-0451524935
3,4,The Great Gatsby,F. Scott Fitzgerald,Tragedy,English,1925,Charles Scribner's Sons,A story of the fabulously wealthy Jay Gatsby a...,180,"classic,wealth,romance,jazz age",4.40,USA,True,NaN,Adult,True,2013.0,978-0743273565
4,5,The Hobbit,J.R.R. Tolkien,Fantasy,English,1937,George Allen & Unwin,"Bilbo Baggins, a hobbit, is swept into an epic...",310,"adventure,dragons,magic",4.75,UK,True,Keith Barker Millennium Book Award,Children,True,2012.0,978-0547928227


# Quelles livres ont été adaptés en film ?

In [4]:
# Sélection des colonnes stratégiques pour l'ETL
# (Ajuste les noms exacts si df_books.columns a révélé des variantes)
cols_interet = ['title', 'author', 'adapted_to_movie', 'movie_release_year']
df_books_filtered = df_books[cols_interet].copy()

# 1. Quelle est la proportion de livres adaptés dans ce catalogue ?
print("--- Répartition des adaptations ---")
print(df_books_filtered['adapted_to_movie'].value_counts(dropna=False))

# 2. Analyse des années de sortie des films
print("\n--- Statistiques sur les années de sortie des films ---")
print(df_books_filtered['movie_release_year'].describe())

--- Répartition des adaptations ---
adapted_to_movie
True     644
False    346
Name: count, dtype: int64

--- Statistiques sur les années de sortie des films ---
count     642.000000
mean     1997.704050
std        22.531925
min      1926.000000
25%      1987.000000
50%      2005.000000
75%      2015.000000
max      2025.000000
Name: movie_release_year, dtype: float64


Dans ce dataset, il y **644 livres qui ont été adaptés en film**.

Les films adaptés de ces livres sont sorti **entre 1926 et 2025**.

Voici quelques un de ces titres adaptés au cinéma :

In [5]:
# On isole uniquement les livres qui ont une adaptation cinéma
df_adaptations = df_books_filtered[df_books_filtered['adapted_to_movie'] == True].dropna(subset=['title'])

print(f"Nombre de livres candidats à la jointure IMDb : {len(df_adaptations)}")
print("\nExemple de titres bruts :")
print(df_adaptations['title'].head(10))

Nombre de livres candidats à la jointure IMDb : 644

Exemple de titres bruts :
0                 Harry Potter and the Sorcerer's Stone
1                                 To Kill a Mockingbird
2                                                  1984
3                                      The Great Gatsby
4                                            The Hobbit
5                                   Pride and Prejudice
7                                     The Da Vinci Code
8                                      The Hunger Games
9                                 The Lord of the Rings
11    The Chronicles of Narnia: The Lion, the Witch ...
Name: title, dtype: object


# Analyse exploratoire approfondie des livres adaptés au cinéma

## Création du sous dataset des livres adaptés au cinéma uniquement

In [9]:
# On filtre immédiatement les livres adaptés
df_adaptations = df_books[df_books['adapted_to_movie'] == True].copy()

print(f"Volume de travail réduit : {df_adaptations.shape[0]} lignes (vs {df_books.shape[0]} au départ).")
print(f"Économie de mémoire : Environs {100 - (len(df_adaptations)/len(df_books)*100):.1f}% de lignes en moins à traiter.")

Volume de travail réduit : 644 lignes (vs 990 au départ).
Économie de mémoire : Environs 34.9% de lignes en moins à traiter.


## Audit des valeurs manquantes

In [10]:
print("--- Valeurs manquantes parmi les livres adaptés ---")
missing_counts = df_adaptations.isnull().sum()
missing_pct = (df_adaptations.isnull().sum() / len(df_adaptations)) * 100

# Création d'un mini tableau de synthèse
df_missing = pd.DataFrame({'Missing Count': missing_counts, 'Percentage (%)': missing_pct.round(2)})
print(df_missing[df_missing['Missing Count'] > 0])

--- Valeurs manquantes parmi les livres adaptés ---
                    Missing Count  Percentage (%)
awards                        417           64.75
movie_release_year              2            0.31


In [14]:
# Isoler et afficher les lignes entières où 'movie_release_year' est manquant
df_nan_years = df_adaptations[df_adaptations['movie_release_year'].isnull()]

print(f"--- Lignes manquantes ({len(df_nan_years)} lignes) ---")
# On utilise display() dans un notebook pour un joli rendu de tableau, ou print() sinon
display(df_nan_years)

--- Lignes manquantes (2 lignes) ---


,id,title,author,genre,language,publicationYear,publisher,description,pageCount,tags,rating_average,most_popular_country,bestseller_status,awards,age_category,adapted_to_movie,movie_release_year,isbn
135,138,Americanah,Chimamanda Ngozi Adichie,Romance,English,2013,Knopf,A young Nigerian woman emigrates to the United...,588,"race,africa,immigration",4.3,Nigeria,True,National Book Critics Circle Award,Adult,True,NaN,978-0307271082
302,307,Green Eggs and Ham,Dr. Seuss,Children's Fiction,English,1960,Random House,Sam-I-Am tries to convince a character to try ...,62,"rhyme,food,kids",4.3,USA,True,NaN,Children,True,NaN,978-0394800165


## Analyse de la cohérence temporelle sur les années de sortie

In [11]:
print("--- Analyse de la colonne 'movie_release_year' ---")
print(df_adaptations['movie_release_year'].describe())

# Top 5 des années les plus représentées
print("\nPériodes les plus actives en adaptations :")
print(df_adaptations['movie_release_year'].value_counts().head(5))

# Est-ce qu'il y a des valeurs aberrantes (ex: < 1895 (début du cinéma) ou > 2026) ?
aberrant_years = df_adaptations[(df_adaptations['movie_release_year'] < 1895) | (df_adaptations['movie_release_year'] > 2026)]
print(f"\nNombre de lignes avec des années aberrantes : {len(aberrant_years)}")

--- Analyse de la colonne 'movie_release_year' ---
count     642.000000
mean     1997.704050
std        22.531925
min      1926.000000
25%      1987.000000
50%      2005.000000
75%      2015.000000
max      2025.000000
Name: movie_release_year, dtype: float64

Périodes les plus actives en adaptations :
movie_release_year
2014.0    22
2015.0    21
2021.0    21
2019.0    21
2018.0    20
Name: count, dtype: int64

Nombre de lignes avec des années aberrantes : 0


## Identification des doublons de clés uniques

Puisque ces livres vont alimenter ta dimension SQL dim_books, chaque livre doit avoir un identifiant unique. Est-ce que le même livre apparaît plusieurs fois dans cette liste d'adaptations ?

In [12]:
# Vérification des doublons sur le titre exact du livre
exact_duplicates_count = df_adaptations.duplicated(subset=['title'], keep=False).sum()
print(f"Nombre de lignes impliquées dans des doublons de titres : {exact_duplicates_count}")

# Si des doublons existent, on regarde un exemple pour comprendre (ex: rééditions, tomes différents...)
if exact_duplicates_count > 0:
    print("\nExemple de doublons détectés :")
    print(df_adaptations[df_adaptations.duplicated(subset=['title'], keep=False)][['title', 'authors', 'movie_release_year']].sort_values(by='title').head(6))

Nombre de lignes impliquées dans des doublons de titres : 0


# Sélection des colonnes pertinentes

In [15]:
# 1. Définition de ta liste de colonnes validées
# (Vérifie juste l'orthographe exacte dans ton dataset : 'author' ou 'authors', 'id' ou 'book_id')
colonnes_validees = ['id', 'title', 'author', 'rating_average', 'movie_release_year', 'isbn']

# 2. Filtrage et copie du dataset
df_books_selection = df_adaptations[colonnes_validees].copy()

# 3. Nettoyage des 2 lignes sans année de film (indispensable pour la future jointure)
df_books_selection = df_books_selection.dropna(subset=['movie_release_year'])

# 4. Conversion de l'année en Entier (le dropna permet de passer de Float à Int proprement)
df_books_selection['movie_release_year'] = df_books_selection['movie_release_year'].astype(int)

print(f"✅ Étape finalisée. Taille du dataset Livres prêt pour l'ETL : {df_books_selection.shape[0]} lignes.")
df_books_selection.head()

✅ Étape finalisée. Taille du dataset Livres prêt pour l'ETL : 642 lignes.


,id,title,author,rating_average,movie_release_year,isbn
0,1,Harry Potter and the Sorcerer's Stone,J.K. Rowling,4.89,2001,978-0747532743
1,2,To Kill a Mockingbird,Harper Lee,4.85,1962,978-0061120084
2,3,1984,George Orwell,4.80,1984,978-0451524935
3,4,The Great Gatsby,F. Scott Fitzgerald,4.40,2013,978-0743273565
4,5,The Hobbit,J.R.R. Tolkien,4.75,2012,978-0547928227


In [16]:
# Recherche du livre spécifique dans le dataset filtré
target_book = "Harry Potter and the Half-Blood Prince"
matches = df_books_selection[df_books_selection['title'].str.contains(target_book, case=False, na=False)]

print(f"--- Résultat de la recherche pour '{target_book}' ---")
if len(matches) > 0:
    print(f"✅ Trouvé ! Nombre de correspondances : {len(matches)}")
    display(matches)
else:
    print("❌ Ce livre n'est pas présent (ou n'est pas marqué comme adapté) dans ton sous-ensemble actuel.")

--- Résultat de la recherche pour 'Harry Potter and the Half-Blood Prince' ---
✅ Trouvé ! Nombre de correspondances : 1


,id,title,author,rating_average,movie_release_year,isbn
594,603,Harry Potter and the Half-Blood Prince,J.K. Rowling,4.57,2009,978-0439784542


# Nettoyage des titres (= clé de jointure avec la table des films)

In [18]:
import re

def prototype_clean_title(title):
    if not isinstance(title, str):
        return ""
    # 1. Passage en minuscules
    title = title.lower()
    # 2. Suppression des parenthèses et crochets (souvent des infos d'édition)
    title = re.sub(r'\(.*\)|\[.*\]', '', title)
    # 3. Suppression de la ponctuation (garde lettres, chiffres et espaces)
    title = re.sub(r'[^a-zA-Z0-9\s]', '', title)
    # 4. Nettoyage des espaces superflus
    title = " ".join(title.split())
    return title

# Test du prototype sur tes données de livres
df_books_selection['cleaned_title_book'] = df_books_selection['title'].apply(prototype_clean_title)

print("--- Comparaison Avant / Après Nettoyage ---")
print(df_books_selection[['title', 'cleaned_title_book']].head(10))

--- Comparaison Avant / Après Nettoyage ---
                                                title  \
0               Harry Potter and the Sorcerer's Stone   
1                               To Kill a Mockingbird   
2                                                1984   
3                                    The Great Gatsby   
4                                          The Hobbit   
5                                 Pride and Prejudice   
7                                   The Da Vinci Code   
8                                    The Hunger Games   
9                               The Lord of the Rings   
11  The Chronicles of Narnia: The Lion, the Witch ...   

                                   cleaned_title_book  
0                harry potter and the sorcerers stone  
1                               to kill a mockingbird  
2                                                1984  
3                                    the great gatsby  
4                                          the h